# 08 - Advanced RAG Patterns (LangGraph)

**Phase 8** of the RAG project. We implement three advanced RAG architectures
using LangGraph for stateful flow control. Each pattern adds self-correction or
adaptive behavior on top of plain retrieval.

### Three patterns

1. **CRAG (Corrective RAG)** - grade retrieved documents for relevance; if too
   many are irrelevant, rewrite the query and retrieve again.

2. **Self-RAG** - the LLM decides at each step: do I need retrieval? Is my
   answer grounded in the documents? Is it useful? Self-corrects if not.

3. **Adaptive RAG** - classify query complexity (simple/moderate/complex) and
   route to a matching pipeline. Complex queries are decomposed into sub-questions.

### Evaluation approach

Because these patterns include generation (not just retrieval), we evaluate on a
small representative sample (2 questions: methodology and results) rather than the
full 5-question retrieval benchmark. We compare:
- Retrieval quality (are the retrieved docs relevant?)
- Answer quality (qualitative inspection)
- Latency and LLM call count
---

## 0. Setup

In [1]:
import json
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.loaders import load_documents_from_directory
from src.ingestion.cleaners import clean_corpus
from src.ingestion.chunkers import chunk_recursive
from src.embeddings.models import create_from_registry
from src.chains.advanced import build_crag, build_self_rag, build_adaptive_rag, run_graph
from notebooks.utils.metrics import load_benchmark_questions, compute_retrieval_metrics

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/hunganh/Code/Python/course_qa_assist


## 1. Corpus, Index and LLM

Same corpus as previous phases: 2,666 core docs, 9,137 chunks,
`mxbai-embed-large` embeddings, and Mistral 7B.

In [2]:
docs = load_documents_from_directory(str(PROJECT_ROOT / "data" / "lectures"))
cleaned_docs, _ = clean_corpus(docs, min_content_length=50)
core_docs = [
    d for d in cleaned_docs
    if "/python/integrations/" not in d.metadata.get("source", "")
]
result = chunk_recursive(core_docs, chunk_size=1000, chunk_overlap=200)
chunks = result.chunks
print(f"Corpus: {len(core_docs)} docs -> {len(chunks)} chunks")

Loaded 2703 document pages from /home/hunganh/Code/Python/course_qa_assist/data/lectures
Corpus: 2666 docs -> 9137 chunks


In [3]:
import chromadb
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

PERSIST_DIR = str(PROJECT_ROOT / "vectorstore" / "chroma_db")
COLLECTION = "advanced_rag_mxbai"
MODELS_YAML = str(PROJECT_ROOT / "configs" / "models.yaml")

embeddings, emb_info = create_from_registry("mxbai_large", config_path=MODELS_YAML)
embeddings.embed_query("warmup")
print(f"Embeddings: {emb_info.model_id}")

client = chromadb.PersistentClient(path=PERSIST_DIR)
try:
    client.delete_collection(COLLECTION)
except Exception:
    pass

print(f"Indexing {len(chunks)} chunks...")
start = time.perf_counter()
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=client,
    collection_name=COLLECTION,
)
print(f"Indexed in {time.perf_counter() - start:.1f}s")

# LLM
llm = ChatOllama(model="mistral:7b", temperature=0.0)
_ = llm.invoke("Hi")  # warmup
print("LLM warmed up.")

Embeddings: mxbai-embed-large
Indexing 9137 chunks...
Indexed in 135.6s
LLM warmed up.


In [4]:
questions = load_benchmark_questions(
    str(PROJECT_ROOT / "data" / "evaluation" / "benchmark_retrieval.json")
)
# Pick one representative question per category for qualitative evaluation
categories = {}
for q in questions:
    categories.setdefault(q.category, []).append(q)

sample_qs = [categories[cat][0] for cat in sorted(categories.keys())]
print(f"Sample questions ({len(sample_qs)}):")
for q in sample_qs:
    print(f"  [{q.category}] {q.query}")

Sample questions (2):
  [methodology] What are the primary characteristics and constraints of TinyML systems mentioned in the text?
  [results] How do large-scale ML systems manage coordination strategies for real-time processing?


---
## 2. CRAG - Corrective RAG

**Flow:**
```
Question -> Retrieve -> Grade docs -> enough relevant? -> Generate
                                   -> too many irrelevant? -> Rewrite query -> Retrieve again
```

The grader uses Mistral 7B to score each doc yes/no for relevance.
If fewer than 50% of docs are relevant, the query is rewritten and retrieval retried
(up to 2 times).

In [5]:
crag_app = build_crag(
    vectorstore=vectorstore,
    llm=llm,
    k=5,
    relevance_threshold=0.5,
    max_rewrites=2,
)
print("CRAG graph compiled.")

CRAG graph compiled.


In [6]:
# Trace CRAG on one question to understand the flow
trace_q = sample_qs[0]  # conceptual
print(f"Tracing CRAG on: '{trace_q.query}'\n")

state = run_graph(crag_app, trace_q.query)

grades = state.get("grades", [])
print(f"Documents retrieved: {len(state.get('documents', []))}")
print(f"Relevance grades: {grades}")
print(f"Relevant fraction: {grades.count('yes') / len(grades):.0%}" if grades else "")
print(f"Query rewrites: {state.get('rewrite_count', 0)}")
print(f"Elapsed: {state['elapsed_ms']:.0f} ms")
print(f"\nGenerated answer:\n{state.get('generation', '')}")

Tracing CRAG on: 'What are the primary characteristics and constraints of TinyML systems mentioned in the text?'



Documents retrieved: 5
Relevance grades: ['yes', 'yes', 'yes', 'yes', 'yes']
Relevant fraction: 100%
Query rewrites: 0
Elapsed: 4861 ms

Generated answer:
 The primary characteristics of TinyML systems, as mentioned in the text, include extreme resource constraints such as limited memory (256KB to 520KB RAM, 1MB to 4MB Flash), low power consumption (0.02-0.25W), and affordability ($35 to $10). The systems are designed for deploying machine learning models on extremely resource-constrained devices like microcontrollers and low-power embedded systems. The primary constraints in TinyML deployments are severe limitations in processing power, memory, and energy consumption. To address these constraints, TinyML frameworks focus on extreme model compression, optimizations for severely constrained environments, and integration with microcontroller-specific architectures.


In [7]:
# Run CRAG on all sample questions
crag_results = []
print(f"CRAG on {len(sample_qs)} questions...")

for q in sample_qs:
    state = run_graph(crag_app, q.query)
    m = compute_retrieval_metrics(
        state.get("documents", []), q, state["elapsed_ms"], k=5
    )
    grades = state.get("grades", [])
    crag_results.append({
        "category": q.category,
        "query": q.query[:60],
        "precision": round(m["precision_at_k"], 3),
        "mrr": round(m["mrr"], 3),
        "rewrites": state.get("rewrite_count", 0),
        "relevant_frac": f"{grades.count('yes')}/{len(grades)}" if grades else "-",
        "latency_ms": round(state["elapsed_ms"]),
    })
    print(f"  [{q.category}] rewrites={state.get('rewrite_count',0)} "
          f"relevant={grades.count('yes') if grades else '?'}/{len(grades)} "
          f"mrr={m['mrr']:.3f} ({state['elapsed_ms']:.0f}ms)")

crag_df = pd.DataFrame(crag_results)
display(crag_df)

CRAG on 2 questions...
  [methodology] rewrites=0 relevant=5/5 mrr=1.000 (4913ms)


  [results] rewrites=0 relevant=5/5 mrr=0.333 (5857ms)


,category,query,precision,mrr,rewrites,relevant_frac,latency_ms
0,methodology,What are the primary characteristics and constraints of Tiny,0.8,1.000,0,5/5,4913
1,results,How do large-scale ML systems manage coordination strategies,0.2,0.333,0,5/5,5857


---
## 3. Self-RAG

**Flow:**
```
Question -> Decide: need retrieval?
         -> Yes: Retrieve -> Generate -> Grounded? -> Yes: Useful? -> Yes: return
                                                                   -> No: rewrite + retry
                                                   -> No: rewrite + retry
         -> No: Generate directly -> Useful? -> Yes: return
                                             -> No: rewrite + retry
```

Three self-reflection checks: retrieval decision, groundedness, usefulness.
Each is an LLM call - making Self-RAG the slowest but most self-aware pattern.

In [8]:
self_rag_app = build_self_rag(
    vectorstore=vectorstore,
    llm=llm,
    k=5,
    max_retries=2,
)
print("Self-RAG graph compiled.")

Self-RAG graph compiled.


In [9]:
# Trace Self-RAG on one question
trace_q = sample_qs[0]
print(f"Tracing Self-RAG on: '{trace_q.query}'\n")

state = run_graph(self_rag_app, trace_q.query)

print(f"Needed retrieval: {state.get('needs_retrieval')}")
print(f"Documents retrieved: {len(state.get('documents', []))}")
print(f"Is grounded: {state.get('is_grounded')}")
print(f"Is useful: {state.get('is_useful')}")
print(f"Retries: {state.get('retry_count', 0)}")
print(f"Elapsed: {state['elapsed_ms']:.0f} ms")
print(f"\nGenerated answer:\n{state.get('generation', '')}")

Tracing Self-RAG on: 'What are the primary characteristics and constraints of TinyML systems mentioned in the text?'

Needed retrieval: True
Documents retrieved: 5
Is grounded: True
Is useful: True
Retries: 0
Elapsed: 4612 ms

Generated answer:
 The primary characteristics of TinyML systems, as mentioned in the text, include extreme resource constraints such as limited memory (256KB to 520KB RAM, 1MB to 4MB Flash), low power consumption (0.02-0.25W), and affordability ($35 to $10). The systems are designed for deploying machine learning models on extremely resource-constrained devices like microcontrollers and low-power embedded systems. The primary constraints in TinyML deployments are severe limitations in processing power, memory, and energy consumption. To address these constraints, TinyML frameworks focus on extreme model compression, optimizations for severely constrained environments, and integration with microcontroller-specific architectures.


In [10]:
self_rag_results = []
print(f"Self-RAG on {len(sample_qs)} questions...")

for q in sample_qs:
    state = run_graph(self_rag_app, q.query)
    m = compute_retrieval_metrics(
        state.get("documents", []), q, state["elapsed_ms"], k=5
    )
    self_rag_results.append({
        "category": q.category,
        "query": q.query[:60],
        "precision": round(m["precision_at_k"], 3),
        "mrr": round(m["mrr"], 3),
        "retrieved": state.get("needs_retrieval"),
        "grounded": state.get("is_grounded"),
        "useful": state.get("is_useful"),
        "retries": state.get("retry_count", 0),
        "latency_ms": round(state["elapsed_ms"]),
    })
    print(f"  [{q.category}] retrieved={state.get('needs_retrieval')} "
          f"grounded={state.get('is_grounded')} useful={state.get('is_useful')} "
          f"retries={state.get('retry_count',0)} mrr={m['mrr']:.3f} ({state['elapsed_ms']:.0f}ms)")

self_rag_df = pd.DataFrame(self_rag_results)
display(self_rag_df)

Self-RAG on 2 questions...
  [methodology] retrieved=True grounded=True useful=True retries=0 mrr=1.000 (4550ms)


  [results] retrieved=True grounded=False useful=True retries=2 mrr=0.000 (15070ms)


,category,query,precision,mrr,retrieved,grounded,useful,retries,latency_ms
0,methodology,What are the primary characteristics and constraints of Tiny,0.8,1.0,True,True,True,0,4550
1,results,How do large-scale ML systems manage coordination strategies,0.0,0.0,True,False,True,2,15070


---
## 4. Adaptive RAG

**Flow:**
```
Question -> Classify complexity
         -> simple:   Retrieve (k=3) -> Generate
         -> moderate: Retrieve (k=5) -> Generate
         -> complex:  Decompose into sub-questions -> Retrieve+Answer each -> Synthesize
```

The key idea: not all questions need the same pipeline. Simple factual questions
don't need 5 docs and a slow model call - complex synthesis questions need
decomposition and multiple retrieval passes.

In [11]:
adaptive_app = build_adaptive_rag(
    vectorstore=vectorstore,
    llm=llm,
    k_simple=3,
    k_moderate=5,
    k_complex=5,
)
print("Adaptive RAG graph compiled.")

Adaptive RAG graph compiled.


In [12]:
# Show complexity classification on all sample questions
print("Complexity classification by Mistral 7B:\n")

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

_COMPLEXITY_PROMPT = ChatPromptTemplate.from_messages([
    (
        "human",
        """Classify the following question into one of three complexity levels:

- "simple": A direct factual question with a single, clear answer.
- "moderate": A question requiring understanding of a concept or process.
- "complex": A multi-part question, comparison, or question requiring synthesis.

Respond with ONLY one word: simple, moderate, or complex.

Question: {question}""",
    ),
])
complexity_chain = _COMPLEXITY_PROMPT | llm | StrOutputParser()

for q in sample_qs:
    c = complexity_chain.invoke({"question": q.query}).strip()
    print(f"  [{q.category}] {c:10} | {q.query}")

Complexity classification by Mistral 7B:

  [methodology] Moderate   | What are the primary characteristics and constraints of TinyML systems mentioned in the text?
  [results] Complex    | How do large-scale ML systems manage coordination strategies for real-time processing?


In [13]:
adaptive_results = []
print(f"Adaptive RAG on {len(sample_qs)} questions...")

for q in sample_qs:
    state = run_graph(adaptive_app, q.query)
    m = compute_retrieval_metrics(
        state.get("documents", []), q, state["elapsed_ms"], k=5
    )
    sub_qs = state.get("sub_questions", [])
    adaptive_results.append({
        "category": q.category,
        "query": q.query[:60],
        "complexity": state.get("complexity", "?"),
        "sub_questions": len(sub_qs),
        "docs_retrieved": len(state.get("documents", [])),
        "precision": round(m["precision_at_k"], 3),
        "mrr": round(m["mrr"], 3),
        "latency_ms": round(state["elapsed_ms"]),
    })
    print(f"  [{q.category}] complexity={state.get('complexity','?')} "
          f"sub_q={len(sub_qs)} docs={len(state.get('documents',[]))} "
          f"mrr={m['mrr']:.3f} ({state['elapsed_ms']:.0f}ms)")

adaptive_df = pd.DataFrame(adaptive_results)
display(adaptive_df)

Adaptive RAG on 2 questions...
  [methodology] complexity=moderate sub_q=0 docs=5 mrr=1.000 (3861ms)
  [results] complexity=moderate sub_q=0 docs=5 mrr=0.333 (5148ms)


,category,query,complexity,sub_questions,docs_retrieved,precision,mrr,latency_ms
0,methodology,What are the primary characteristics and constraints of Tiny,moderate,0,5,0.8,1.000,3861
1,results,How do large-scale ML systems manage coordination strategies,moderate,0,5,0.2,0.333,5148


In [14]:
# Show a complex query trace in detail
complex_q = "Compare the different preference optimization methods proposed in TokenRatio and explain when to use each one."
print(f"Complex query: '{complex_q}'\n")

state = run_graph(adaptive_app, complex_q)

print(f"Complexity: {state.get('complexity')}")
print(f"Sub-questions:")
for i, sq in enumerate(state.get("sub_questions", []), 1):
    print(f"  {i}. {sq}")
print(f"\nDocs retrieved: {len(state.get('documents', []))}")
print(f"Elapsed: {state['elapsed_ms']:.0f} ms")
print(f"\nSynthesized answer:\n{state.get('generation', '')}")

Complex query: 'Compare the different preference optimization methods proposed in TokenRatio and explain when to use each one.'

Complexity: complex
Sub-questions:
  1. List the preference optimization methods proposed in TokenRatio.
  2. Describe each of the preference optimization methods proposed in TokenRatio.
  3. Under what circumstances would it be appropriate to use each of the preference optimization methods proposed in TokenRatio?

Docs retrieved: 1
Elapsed: 20029 ms

Synthesized answer:
 The TokenRatio paper proposes two preference optimization methods based on the token-level Bradley–Terry model: TBPO-Q and TBPO-A. Both methods aim to optimize token-level preferences via ratio matching for principled learning algorithms with theoretical guarantees.

1. TBPO-Q uses the reference-policy state-action value as the score function S, defined as S(s, a) = Q πref (s, a). The preference signal is provided throughout generation, encouraging extended prefixes to separate in quality as

---
## 5. Global Comparison

In [15]:
# Baseline: plain similarity (no LangGraph)
from src.retrieval.reranker import retrieve_no_reranking

baseline_results = []
for q in sample_qs:
    res = retrieve_no_reranking(vectorstore, q.query, k=5)
    m = compute_retrieval_metrics(res.docs, q, res.elapsed_ms, k=5)
    baseline_results.append({
        "category": q.category,
        "precision": round(m["precision_at_k"], 3),
        "mrr": round(m["mrr"], 3),
        "latency_ms": round(res.elapsed_ms),
    })
baseline_df = pd.DataFrame(baseline_results)

# Summary table
def summarize(df, label, llm_calls_per_q):
    return {
        "pattern": label,
        "avg_precision": round(df["precision"].mean(), 3),
        "avg_mrr": round(df["mrr"].mean(), 3),
        "avg_latency_ms": round(df["latency_ms"].mean()),
        "llm_calls_per_q": llm_calls_per_q,
    }

# Estimate LLM calls per question
# CRAG: 5 (grade) + 1 (generate) = 6, +3 if rewrite
# Self-RAG: 1 (decide) + 1 (generate) + 1 (groundedness) + 1 (usefulness) = 4
# Adaptive: 1 (classify) + 1-3 (generate / sub answers + synthesize)
summary_data = [
    summarize(baseline_df, "baseline (similarity)", 0),
    summarize(crag_df, "CRAG", "5+1"),
    summarize(self_rag_df, "Self-RAG", 4),
    summarize(adaptive_df, "Adaptive RAG", "1-4"),
]

summary_df = pd.DataFrame(summary_data)
print("Global comparison (sample, n=4 questions per category):")
display(summary_df)

Global comparison (sample, n=4 questions per category):


,pattern,avg_precision,avg_mrr,avg_latency_ms,llm_calls_per_q
0,baseline (similarity),0.5,0.666,32,0
1,CRAG,0.5,0.666,5385,5+1
2,Self-RAG,0.4,0.500,9810,4
3,Adaptive RAG,0.5,0.666,4504,1-4


---
## 6. Qualitative Answer Inspection

Retrieval metrics don't capture answer quality. We compare answers from all
three patterns on one representative question.

In [16]:
compare_q = sample_qs[0]  # conceptual question
print(f"Question: {compare_q.query}\n")
print("=" * 70)

for label, app in [("CRAG", crag_app), ("Self-RAG", self_rag_app), ("Adaptive RAG", adaptive_app)]:
    state = run_graph(app, compare_q.query)
    print(f"\n--- {label} ({state['elapsed_ms']:.0f} ms) ---")
    print(state.get("generation", "(no generation)"))
    print()

Question: What are the primary characteristics and constraints of TinyML systems mentioned in the text?


--- CRAG (4689 ms) ---
 The primary characteristics of TinyML systems, as mentioned in the text, include extreme resource constraints such as limited memory (256KB to 520KB RAM, 1MB to 4MB Flash), low power consumption (0.02-0.25W), and affordability ($35 to $10). The systems are designed for deploying machine learning models on extremely resource-constrained devices like microcontrollers and low-power embedded systems. The primary constraints in TinyML deployments are severe limitations in processing power, memory, and energy consumption. To address these constraints, TinyML frameworks focus on extreme model compression, optimizations for severely constrained environments, and integration with microcontroller-specific architectures.


--- Self-RAG (4319 ms) ---
 The primary characteristics of TinyML systems, as mentioned in the text, include extreme resource constraints such as li

---
## 7. Save Results

In [17]:
results_output = {
    "corpus": {
        "num_core_docs": len(core_docs),
        "num_chunks": len(chunks),
        "embedding_model": emb_info.model_id,
    },
    "evaluation": {
        "note": "Sample evaluation (1 question per category, n=4)",
        "questions": [q.query for q in sample_qs],
    },
    "patterns": {
        "baseline": baseline_df[["category", "precision", "mrr", "latency_ms"]].to_dict(orient="records"),
        "crag": crag_df[["category", "precision", "mrr", "rewrites", "latency_ms"]].to_dict(orient="records"),
        "self_rag": self_rag_df[["category", "precision", "mrr", "retries", "latency_ms"]].to_dict(orient="records"),
        "adaptive_rag": adaptive_df[["category", "complexity", "precision", "mrr", "latency_ms"]].to_dict(orient="records"),
    },
    "summary": summary_df.to_dict(orient="records"),
}

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)
output_path = results_dir / "advanced_rag_comparison.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results_output, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_path}")

Results saved to /home/hunganh/Code/Python/course_qa_assist/results/advanced_rag_comparison.json


---
## 8. Cleanup

In [18]:
try:
    client.delete_collection(COLLECTION)
    print(f"Collection '{COLLECTION}' deleted.")
except Exception:
    pass
print(f"Remaining collections: {[c.name for c in client.list_collections()]}")

Collection 'advanced_rag_mxbai' deleted.
Remaining collections: ['course_qa_naive']


---
## 9. Summary and Conclusions

### Results (sample, n=2 questions: methodology and results)

| Pattern | Avg Precision | Avg MRR | Avg Latency | LLM calls/q |
|---|---|---|---|---|
| Baseline (similarity) | 0.500 | 0.666 | 32 ms | 0 |
| CRAG | 0.500 | 0.666 | 5385 ms | 5+1 |
| Self-RAG | 0.400 | 0.500 | 9810 ms | 4 |
| Adaptive RAG | 0.500 | 0.666 | 4504 ms | 1-4 |

### MRR by category

| Category | Baseline | CRAG | Self-RAG | Adaptive |
|---|---|---|---|---|
| methodology | 1.000 | 1.000 | 1.000 | 1.000 |
| results | 0.333 | 0.333 | 0.000 | 0.333 |

### Key Takeaways

1. **CRAG and Adaptive RAG match baseline on this sample.** Avg MRR is 0.666 and
   avg precision is 0.500 for all three, with CRAG showing 0 rewrites on both questions.

2. **Self-RAG underperforms on the results question.** It hits MRR 0.000 with 2 retries
   and the highest latency (9.8 s), pulling average MRR down to 0.500.

3. **Adaptive RAG never triggers the complex path.** Both questions were classified as
   `moderate`, so it behaves like standard retrieval with extra overhead.

4. **Latency cost dominates.** Baseline is ~32 ms, while CRAG is ~5.4 s, Adaptive is
   ~4.5 s, and Self-RAG is ~9.8 s per question.

5. **Sample size is tiny.** Re-run on a larger set before generalizing.

### When to use each pattern

| Pattern | Use when | Requires |
|---|---|---|
| CRAG | Retrieval quality is uncertain and you can afford ~5-6 s per query | Any LLM that can grade relevance |
| Self-RAG | You want self-reflection and can tolerate high latency | Stronger LLM for reliable self-evaluation |
| Adaptive RAG | Query complexity varies and the classifier is reliable | LLM that distinguishes complexity levels |
| Baseline | Low-latency required; corpus well-indexed | None |

### Next Step: Phase 9 - Evaluation with RAGAS

Build a comprehensive end-to-end evaluation pipeline using RAGAS metrics:
faithfulness, answer relevancy, context precision, context recall.
This measures full answer quality beyond retrieval-only MRR.